In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Helper Functions

In [36]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    # Since the Wind Degree is scrapped from the wind icon in the table, 
    # if the wind speed is too low, the icon will just be a circle without any rotation. 
    # Hence, they should be replaced by the corresponding degree of the Wind Direction.
    # Wind degree from timeanddate webpage
    wind_deg_dict = {
        'N': 90, 'NNE': 110, 'NE': 130, 'ENE': 150,
        'E': 180, 'ESE': 200, 'SE': 220, 'SSE': 240,
        'S': 270, 'SSW': 290, 'SW': 310, 'WSW': 330,
        'W': 360, 'WNW': 380, 'NW': 400, 'NNW': 430,
        np.nan: np.nan # Handle NaN values in Wind Direction
    }

    df['Wind Degree'] = df['Wind Degree'].fillna(df['Wind Direction'].map(wind_deg_dict)) # Fill NA by Wind Direction
    
    # Standardize the Wind Degree
    df['Wind Degree'] = df['Wind Degree'] - 90

    # Rename Barometer to Air Pressure
    df = df.rename({'Barometer': 'Air Pressure'}, axis=1)
    
    # Adding time related columns
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce') # Convert to datetime, coerce errors to NaT
    df['Day'] = df['Date'].dt.day
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    df['Week Number'] = df['Date'].dt.isocalendar().week
    
    # Add temperature difference
    df['Temp Diff'] = df['Temp High'] - df['Temp Low']

    # Add average temperature
    df['Average Temp'] = round((df['Temp High'] + df['Temp Low']) / 2, 1)
    
    return df

# Forecast Weather

In [37]:
forecast_df = pd.read_csv('csv/forecast_weather_data.csv')
forecast_df.head()

,Weekday,Date,Temp High,Temp Low,Condition,Feels Like,Humidity,Precipitation_Rain,Precipitation_Snow,Precipitation Chance,Wind Direction,Wind Degree,Wind Speed,City
0,Wed,May 27,65.0,54.0,Passing showers. Mostly cloudy,65.0,55.0,0.03,0.0,24.0,SSW,290.0,10.0,Los Angeles
1,Thu,May 28,68.0,54.0,Showers late. Broken clouds,66.0,51.0,0.00,0.0,21.0,SW,320.0,12.0,Los Angeles
2,Fri,May 29,69.0,54.0,Mostly sunny,68.0,51.0,0.00,0.0,5.0,SW,320.0,12.0,Los Angeles
3,Sat,May 30,74.0,55.0,Sunny,77.0,51.0,0.00,0.0,0.0,SW,320.0,11.0,Los Angeles
4,Sun,May 31,80.0,57.0,Sunny,79.0,43.0,0.00,0.0,0.0,SW,320.0,10.0,Los Angeles


In [38]:
forecast_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Weekday               45 non-null     object 
 1   Date                  45 non-null     object 
 2   Temp High             45 non-null     float64
 3   Temp Low              45 non-null     float64
 4   Condition             45 non-null     object 
 5   Feels Like            45 non-null     float64
 6   Humidity              45 non-null     float64
 7   Precipitation_Rain    45 non-null     float64
 8   Precipitation_Snow    45 non-null     float64
 9   Precipitation Chance  45 non-null     float64
 10  Wind Direction        45 non-null     object 
 11  Wind Degree           45 non-null     float64
 12  Wind Speed            45 non-null     float64
 13  City                  45 non-null     object 
dtypes: float64(9), object(5)
memory usage: 5.1+ KB


## Weekday

In [39]:
forecast_df['Weekday'] = forecast_df['Weekday'].map({
    'Sun': 'Sunday',
    'Mon': 'Monday',
    'Tue': 'Tuesday',
    'Wed': 'Wednesday',
    'Thu': 'Thursday',
    'Fri': 'Friday',
    'Sat': 'Saturday'
})
forecast_df.head()

,Weekday,Date,Temp High,Temp Low,Condition,Feels Like,Humidity,Precipitation_Rain,Precipitation_Snow,Precipitation Chance,Wind Direction,Wind Degree,Wind Speed,City
0,Wednesday,May 27,65.0,54.0,Passing showers. Mostly cloudy,65.0,55.0,0.03,0.0,24.0,SSW,290.0,10.0,Los Angeles
1,Thursday,May 28,68.0,54.0,Showers late. Broken clouds,66.0,51.0,0.00,0.0,21.0,SW,320.0,12.0,Los Angeles
2,Friday,May 29,69.0,54.0,Mostly sunny,68.0,51.0,0.00,0.0,5.0,SW,320.0,12.0,Los Angeles
3,Saturday,May 30,74.0,55.0,Sunny,77.0,51.0,0.00,0.0,0.0,SW,320.0,11.0,Los Angeles
4,Sunday,May 31,80.0,57.0,Sunny,79.0,43.0,0.00,0.0,0.0,SW,320.0,10.0,Los Angeles


## Date

In [40]:
forecast_df['Date'] = forecast_df['Date'] + ', 2026'
forecast_df.head()

,Weekday,Date,Temp High,Temp Low,Condition,Feels Like,Humidity,Precipitation_Rain,Precipitation_Snow,Precipitation Chance,Wind Direction,Wind Degree,Wind Speed,City
0,Wednesday,"May 27, 2026",65.0,54.0,Passing showers. Mostly cloudy,65.0,55.0,0.03,0.0,24.0,SSW,290.0,10.0,Los Angeles
1,Thursday,"May 28, 2026",68.0,54.0,Showers late. Broken clouds,66.0,51.0,0.00,0.0,21.0,SW,320.0,12.0,Los Angeles
2,Friday,"May 29, 2026",69.0,54.0,Mostly sunny,68.0,51.0,0.00,0.0,5.0,SW,320.0,12.0,Los Angeles
3,Saturday,"May 30, 2026",74.0,55.0,Sunny,77.0,51.0,0.00,0.0,0.0,SW,320.0,11.0,Los Angeles
4,Sunday,"May 31, 2026",80.0,57.0,Sunny,79.0,43.0,0.00,0.0,0.0,SW,320.0,10.0,Los Angeles


## Clean Data

In [41]:
forecast_df = clean_data(forecast_df)
forecast_df.head()

,Weekday,Date,Temp High,Temp Low,Condition,Feels Like,Humidity,Precipitation_Rain,Precipitation_Snow,Precipitation Chance,Wind Direction,Wind Degree,Wind Speed,City,Day,Month,Year,Week Number,Temp Diff,Average Temp
0,Wednesday,2026-05-27,65.0,54.0,Passing showers. Mostly cloudy,65.0,55.0,0.03,0.0,24.0,SSW,200.0,10.0,Los Angeles,27.0,5.0,2026.0,22,11.0,59.5
1,Thursday,2026-05-28,68.0,54.0,Showers late. Broken clouds,66.0,51.0,0.00,0.0,21.0,SW,230.0,12.0,Los Angeles,28.0,5.0,2026.0,22,14.0,61.0
2,Friday,2026-05-29,69.0,54.0,Mostly sunny,68.0,51.0,0.00,0.0,5.0,SW,230.0,12.0,Los Angeles,29.0,5.0,2026.0,22,15.0,61.5
3,Saturday,2026-05-30,74.0,55.0,Sunny,77.0,51.0,0.00,0.0,0.0,SW,230.0,11.0,Los Angeles,30.0,5.0,2026.0,22,19.0,64.5
4,Sunday,2026-05-31,80.0,57.0,Sunny,79.0,43.0,0.00,0.0,0.0,SW,230.0,10.0,Los Angeles,31.0,5.0,2026.0,22,23.0,68.5


# Past Weather

In [42]:
past_df = pd.read_csv('csv/past_weather_data.csv')
past_df.head()

,Weekday,Date,Time,Temp High,Temp Low,Condition,Humidity,Barometer,Wind Direction,Wind Degree,Wind Speed,City
0,Friday,"May 1, 2026",12:00 am — 6:00 am,63.0,63.0,Overcast,76.0,29.93,SSE,250.0,4.350,Los Angeles
1,Friday,"May 1, 2026",6:00 am — 12:00 pm,68.0,63.0,Overcast,68.0,29.97,ESE,200.0,3.728,Los Angeles
2,Friday,"May 1, 2026",12:00 pm — 6:00 pm,70.0,66.0,Sunny,62.0,29.90,W,350.0,11.185,Los Angeles
3,Friday,"May 1, 2026",6:00 pm — 12:00 am,64.0,61.0,Passing clouds,79.0,29.93,WNW,380.0,6.836,Los Angeles
4,Saturday,"May 2, 2026",12:00 am — 6:00 am,61.0,59.0,Low clouds,85.0,29.93,WSW,330.0,3.728,Los Angeles


In [43]:
past_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Weekday         1739 non-null   object 
 1   Date            1739 non-null   object 
 2   Time            1739 non-null   object 
 3   Temp High       1736 non-null   float64
 4   Temp Low        1736 non-null   float64
 5   Condition       1736 non-null   object 
 6   Humidity        1736 non-null   float64
 7   Barometer       1736 non-null   float64
 8   Wind Direction  1736 non-null   object 
 9   Wind Degree     1579 non-null   float64
 10  Wind Speed      1736 non-null   float64
 11  City            1739 non-null   object 
dtypes: float64(6), object(6)
memory usage: 163.2+ KB


## Clean Data

In [44]:
past_df = clean_data(past_df)
past_df.head()

,Weekday,Date,Time,Temp High,Temp Low,Condition,Humidity,Air Pressure,Wind Direction,Wind Degree,Wind Speed,City,Day,Month,Year,Week Number,Temp Diff,Average Temp
0,Friday,2026-05-01,12:00 am — 6:00 am,63.0,63.0,Overcast,76.0,29.93,SSE,160.0,4.350,Los Angeles,1,5,2026,18,0.0,63.0
1,Friday,2026-05-01,6:00 am — 12:00 pm,68.0,63.0,Overcast,68.0,29.97,ESE,110.0,3.728,Los Angeles,1,5,2026,18,5.0,65.5
2,Friday,2026-05-01,12:00 pm — 6:00 pm,70.0,66.0,Sunny,62.0,29.90,W,260.0,11.185,Los Angeles,1,5,2026,18,4.0,68.0
3,Friday,2026-05-01,6:00 pm — 12:00 am,64.0,61.0,Passing clouds,79.0,29.93,WNW,290.0,6.836,Los Angeles,1,5,2026,18,3.0,62.5
4,Saturday,2026-05-02,12:00 am — 6:00 am,61.0,59.0,Low clouds,85.0,29.93,WSW,240.0,3.728,Los Angeles,2,5,2026,18,2.0,60.0


## Encoding time of day

In [45]:
# Rename time of day
past_df['Time'] = past_df['Time'].map(
    {
        '12:00 am — 6:00 am': 1,
        '6:00 am — 12:00 pm': 2,
        '12:00 pm — 6:00 pm': 3,
        '6:00 pm — 12:00 am': 4
    }, 
)
past_df = past_df.rename(columns={'Time': 'Time of Day'})
past_df.head()

,Weekday,Date,Time of Day,Temp High,Temp Low,Condition,Humidity,Air Pressure,Wind Direction,Wind Degree,Wind Speed,City,Day,Month,Year,Week Number,Temp Diff,Average Temp
0,Friday,2026-05-01,1,63.0,63.0,Overcast,76.0,29.93,SSE,160.0,4.350,Los Angeles,1,5,2026,18,0.0,63.0
1,Friday,2026-05-01,2,68.0,63.0,Overcast,68.0,29.97,ESE,110.0,3.728,Los Angeles,1,5,2026,18,5.0,65.5
2,Friday,2026-05-01,3,70.0,66.0,Sunny,62.0,29.90,W,260.0,11.185,Los Angeles,1,5,2026,18,4.0,68.0
3,Friday,2026-05-01,4,64.0,61.0,Passing clouds,79.0,29.93,WNW,290.0,6.836,Los Angeles,1,5,2026,18,3.0,62.5
4,Saturday,2026-05-02,1,61.0,59.0,Low clouds,85.0,29.93,WSW,240.0,3.728,Los Angeles,2,5,2026,18,2.0,60.0


# Climate

In [46]:
climate_df = pd.read_csv('csv/climate_data.csv')
climate_df.head()

,Month,High Temp,Low Temp,Mean Temp,Precipitation,Humidity,Dew Point,Wind,Pressure,Visibility,City
0,January,69.0,49.0,59.0,3.37,59.0,42.0,3.0,30.08,10.0,Los Angeles
1,February,68.0,50.0,59.0,3.57,62.0,44.0,4.0,30.05,10.0,Los Angeles
2,March,70.0,53.0,62.0,2.09,64.0,47.0,4.0,30.02,9.0,Los Angeles
3,April,73.0,55.0,64.0,0.69,64.0,49.0,5.0,29.98,10.0,Los Angeles
4,May,74.0,59.0,66.0,0.35,68.0,54.0,5.0,29.94,9.0,Los Angeles


In [47]:
climate_df['Temp Diff'] = climate_df['High Temp'] - climate_df['Low Temp']
climate_df.head()

,Month,High Temp,Low Temp,Mean Temp,Precipitation,Humidity,Dew Point,Wind,Pressure,Visibility,City,Temp Diff
0,January,69.0,49.0,59.0,3.37,59.0,42.0,3.0,30.08,10.0,Los Angeles,20.0
1,February,68.0,50.0,59.0,3.57,62.0,44.0,4.0,30.05,10.0,Los Angeles,18.0
2,March,70.0,53.0,62.0,2.09,64.0,47.0,4.0,30.02,9.0,Los Angeles,17.0
3,April,73.0,55.0,64.0,0.69,64.0,49.0,5.0,29.98,10.0,Los Angeles,18.0
4,May,74.0,59.0,66.0,0.35,68.0,54.0,5.0,29.94,9.0,Los Angeles,15.0


# Save to DB

In [48]:
import sqlite3

In [49]:
# Connect to the database
try:
    with sqlite3.connect("db/weather.db") as conn:
        cursor = conn.cursor()

except sqlite3.Error as e:
    print(f"An error occurred while connecting to the database: {e}")

In [50]:
forecast_df.to_sql('Forecast_Weather', conn, if_exists='replace')
past_df.to_sql('Past_Weather', conn, if_exists='replace')
climate_df.to_sql('Climate', conn, if_exists='replace')

36